# 30 · Scale-Up Experiments (Reviewer Fixes #7–9)

Addresses the three "needs modest additional work" items from review feedback:

- **#7** — TMR real-channel trials: `n=20 -> n=100` (tighter Wilson CI on the 0%-correction finding)
- **#8** — A second FABRIC site pair with different latency/congestion, to test whether "real network doesn't matter for silent divergence" holds beyond one low-noise link
- **#9** — 5 more real key pairs (5 -> 10), to de-risk Table 2 (finite-key length) from being sensitive to any one key

Each section is self-contained — run whichever you need. Sections B and C provision/reuse a
live FABRIC slice and will take a while (network round-trips); start them and let them run.

**Prerequisite:** existing repo checked out with `results/` populated from your original runs
(this notebook reads several of your existing CSVs to append to / compare against them).

## Section A — TMR: extend n=20 → n=100 (Feedback #7)

Reuses the exact `tmr_real_channel_trial` mechanism from
`notebooks/mitigation_v2_redundant_pa_tmr.ipynb`, at the same `fault_prob=0.3`, same key
(key0), just more trials. Appends to your existing `results/mitigation_tmr_realchannel.csv`
rather than overwriting it, so the first 20 trials you already collected are preserved.

In [1]:
import sys, json, re, time
from pathlib import Path
import numpy as np
import pandas as pd
from galois import GF2
from randextract import ToeplitzHashing

PROJECT_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(PROJECT_DIR))
sys.path.insert(0, str(PROJECT_DIR / 'scripts'))

import deploy_fabric as deploy
from qne.cascade import Key, ORIGINAL, Reconciliation, MockClassicalSession
from qne.cascade.key import key_from_sifted_json

RESULTS = PROJECT_DIR / "results"
print("Imports complete")


Imports complete


In [2]:
# --- Connect to your existing slice and re-sync key0 (same as mitigation_v2 notebook) ---
SLICE_NAME = 'qfabric-bb84-2'

fablib = deploy.get_fablib()
slice_obj = fablib.get_slice(name=SLICE_NAME)
slice_obj.show()

alice = slice_obj.get_node("alice")
bob = slice_obj.get_node("bob")
bob_ip = "10.10.1.2"

GEN_ALICE = RESULTS / "data" / "fabric_alice_sifted_bits_genonly.json"
GEN_BOB = RESULTS / "data" / "fabric_bob_sifted_bits_genonly.json"
META_PATH = RESULTS / "data" / "fabric_key0_pe_meta.json"

alice_key, alice_indices = key_from_sifted_json(str(GEN_ALICE), "alice_bits")
bob_key, bob_indices = key_from_sifted_json(str(GEN_BOB), "bob_bits")
assert alice_indices == bob_indices

meta = json.loads(META_PATH.read_text())
k_pe, real_qber = meta["k"], meta["qber"]
print(f"key0 loaded: n={alice_key.get_nr_bits()} bits, k={k_pe}, qber={real_qber:.4f}")

alice.upload_file(str(GEN_ALICE), "qfabric/results/alice_sifted_bits.json")
bob.upload_file(str(GEN_ALICE), "qfabric/results/alice_sifted_bits.json")
bob.upload_file(str(GEN_BOB), "qfabric/results/bob_sifted_bits.json")
print("Remote nodes re-synced to key0")


Orchestrator,orchestrator.fabric-testbed.net
Credential Manager,cm.fabric-testbed.net
Core API,uis.fabric-testbed.net
Artifact Manager,artifacts.fabric-testbed.net
CEPH Manager,https://ceph-mgr.fabric-testbed.net
Token File,/home/fabric/work/fabric_config/id_token.json
Project ID,24f4c8f3-e872-492a-9a83-b48211a91966
Bastion Host,bastion.fabric-testbed.net
Bastion Username,audreyf_0000527467
Bastion Private Key File,/home/fabric/work/fabric_config/fabric_bastion_key
Slice Public Key File,/home/fabric/work/fabric_config/slice_key.pub


User: audreyf@illinois.edu bastion key is valid!
Configuration is valid


SliceNotFoundError: 'Unable to find slice "qfabric-bb84-2" for this project. Check slice name spelling and project id.'

In [ ]:
def extract_output_path(stdout_text, marker):
    match = re.search(rf"{marker} (\S+\.json)", stdout_text)
    return match.group(1) if match else None


def tmr_real_channel_trial(bob, alice, bob_ip, real_qber, run, base_seed, fault_prob, k,
                            n_replicas=3, timeout=180):
    """Unchanged from mitigation_v2_redundant_pa_tmr.ipynb -- copied here so this
    notebook doesn't depend on that one having been run first in the same kernel."""
    replica_results = []
    for rep in range(n_replicas):
        seed = base_seed + rep * 1000
        bob_output = f"results/bob_tmr_run{run}_rep{rep}.json"

        bob_thread = bob.execute_thread(
            f"cd ~/qfabric && ~/qfabric/.venv/bin/python3 scripts/bob_cascade_driver.py "
            f"--key-json results/bob_sifted_bits.json --alice-key-json results/alice_sifted_bits.json "
            f"--host {bob_ip} --port 5200 --qber {real_qber} --k {k} --seed {seed} "
            f"--reconciliation-prob {fault_prob} --output {bob_output}"
        )
        alice_thread = alice.execute_thread(
            f"cd ~/qfabric && ~/qfabric/.venv/bin/python3 scripts/alice_cascade_responder.py "
            f"--key-json results/alice_sifted_bits.json --bob-host {bob_ip} --port 5200 "
            f"--seed {seed} --output results/alice_tmr_run{run}_rep{rep}.json"
        )

        bob_out = bob_thread.result(timeout=timeout)
        try:
            alice_thread.result(timeout=timeout // 2)
        except Exception:
            pass

        bob_output_path = extract_output_path(bob_out[0], "Result written to")
        if bob_output_path is None:
            replica_results.append({"status": "crashed", "bits": None})
            continue

        stdout, _ = bob.execute(f"cat ~/qfabric/{bob_output_path}", quiet=True)
        try:
            data = json.loads(stdout)
        except json.JSONDecodeError:
            replica_results.append({"status": "no_output", "bits": None})
            continue

        if data.get("non_convergent"):
            replica_results.append({"status": "non_convergent", "bits": None})
        elif "bob_reconciled_bits" in data:
            replica_results.append({
                "status": "converged", "bits": np.array(data["bob_reconciled_bits"]),
                "remaining_errors": data.get("remaining_errors_after_reconciliation"),
            })
        else:
            replica_results.append({"status": "converged_no_bits_saved", "bits": None})

        time.sleep(2)
    return replica_results

print("tmr_real_channel_trial defined")


In [ ]:
# --- Run 80 more trials (indices 20..99), continuing seeds from where the
# original 20 left off, and append to the SAME csv the paper's Table/Fig 9 was built from ---
TMR_CSV = RESULTS / "data" / "mitigation_tmr_realchannel.csv"
existing_tmr = pd.read_csv(str(TMR_CSV))
start_run = int(existing_tmr["run"].max()) + 1  # continues at 20 if you have the original 20
N_NEW_TRIALS = 80  # -> 100 total; raise/lower as time allows, CI tightens with more

new_rows = []
for run in range(start_run, start_run + N_NEW_TRIALS):
    base_seed = 42 + run * 10
    replicas = tmr_real_channel_trial(bob, alice, bob_ip, real_qber, run, base_seed,
                                       fault_prob=0.3, k=k_pe)
    statuses = [r["status"] for r in replicas]
    valid = [r["bits"] for r in replicas if r["bits"] is not None]

    if len(valid) < 2:
        new_rows.append({"run": run, "statuses": statuses, "result": "insufficient_valid_replicas",
                          "n_valid": len(valid), "corrected": False})
        print(f"run {run}: insufficient valid replicas ({statuses})")
        continue

    majority = (np.stack(valid).sum(axis=0) > len(valid) / 2).astype(np.uint8)
    n_errors = alice_key.nr_bits_different(Key(bits=majority))
    corrected = n_errors == 0
    new_rows.append({"run": run, "statuses": statuses, "result": "voted",
                      "n_valid": len(valid), "n_errors_after_tmr": n_errors, "corrected": corrected})
    print(f"run {run}: n_valid={len(valid)}, errors_after_vote={n_errors}, corrected={corrected}")

    # Checkpoint after every trial -- real-channel runs are slow and this can be
    # interrupted/resumed safely since start_run is read from the file each time.
    df_checkpoint = pd.concat([existing_tmr, pd.DataFrame(new_rows)], ignore_index=True)
    df_checkpoint.to_csv(str(TMR_CSV), index=False)

print(f"\nDone. Total trials in {TMR_CSV.name}: {len(df_checkpoint)}")


In [ ]:
# --- Recompute the Wilson 95% CI on the correction rate at n=100 (replaces the
# n=20, [0%, 16.1%] CI reported in the submitted draft) ---
from scipy.stats import norm

def wilson_ci(successes, n, z=1.96):
    if n == 0:
        return np.nan, np.nan, np.nan
    p_hat = successes / n
    denom = 1 + z**2 / n
    center = (p_hat + z**2 / (2 * n)) / denom
    halfwidth = (z / denom) * np.sqrt(p_hat * (1 - p_hat) / n + z**2 / (4 * n**2))
    return p_hat, max(0, center - halfwidth), min(1, center + halfwidth)

df_final = pd.read_csv(str(TMR_CSV))
voted = df_final[df_final["result"] == "voted"]
n_trials = len(voted)
n_corrected = int(voted["corrected"].sum())
p_hat, lo, hi = wilson_ci(n_corrected, n_trials)

mean_errors = voted["n_errors_after_tmr"].mean()
se_errors = voted["n_errors_after_tmr"].std() / np.sqrt(n_trials)

print(f"n={n_trials} real-channel TMR trials")
print(f"corrected: {n_corrected}/{n_trials}  (p_hat={p_hat:.3f})")
print(f"95% Wilson CI on true correction rate: [{lo*100:.1f}%, {hi*100:.1f}%]")
print(f"mean residual errors after vote: {mean_errors:.2f} +/- {se_errors:.2f} (SE)")


In [1]:
import sys, json
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(PROJECT_DIR))
sys.path.insert(0, str(PROJECT_DIR / 'scripts'))

import deploy_fabric as deploy
from qne.config import ScenarioConfig
from qne.cascade.sweep_utils import run_real_channel_trial, collect_key_pairs

RESULTS = PROJECT_DIR / "results"
fablib = deploy.get_fablib()

# --- Sanity-check capacity before provisioning; adjust SITE_BOB2 based on what's free ---
for site in ["TOKY", "UTAH", "FIU", "UCSD", "MICH", "CLEM", "STAR", "SALT"]:
    try:
        print(site, fablib.get_site_attribute(site, "cores_available"))
    except Exception as e:
        print(site, "unavailable:", e)


Orchestrator,orchestrator.fabric-testbed.net
Credential Manager,cm.fabric-testbed.net
Core API,uis.fabric-testbed.net
Artifact Manager,artifacts.fabric-testbed.net
CEPH Manager,https://ceph-mgr.fabric-testbed.net
Token File,/home/fabric/work/fabric_config/id_token.json
Project ID,24f4c8f3-e872-492a-9a83-b48211a91966
Bastion Host,bastion.fabric-testbed.net
Bastion Username,audreyf_0000527467
Bastion Private Key File,/home/fabric/work/fabric_config/fabric_bastion_key
Slice Public Key File,/home/fabric/work/fabric_config/slice_key.pub


TOKY unavailable: 'FablibManager' object has no attribute 'get_site_attribute'
UTAH unavailable: 'FablibManager' object has no attribute 'get_site_attribute'
FIU unavailable: 'FablibManager' object has no attribute 'get_site_attribute'
UCSD unavailable: 'FablibManager' object has no attribute 'get_site_attribute'
MICH unavailable: 'FablibManager' object has no attribute 'get_site_attribute'
CLEM unavailable: 'FablibManager' object has no attribute 'get_site_attribute'
STAR unavailable: 'FablibManager' object has no attribute 'get_site_attribute'
SALT unavailable: 'FablibManager' object has no attribute 'get_site_attribute'


In [ ]:
import json

fablib = deploy.get_fablib()
slice_obj = fablib.get_slice(name='qfabric-bb84-2')
bob = slice_obj.get_node("bob")

known_metadata = [
    {"index": 0, "m_sifted": 3894, "k_pe": 390, "n_bits": 3504, "qber": 0.01282051282051282},
    {"index": 1, "m_sifted": 3788, "k_pe": 379, "n_bits": 3409, "qber": 0.0158311345646438},
    {"index": 2, "m_sifted": 3734, "k_pe": 374, "n_bits": 3360, "qber": 0.00267379679144385},
    {"index": 3, "m_sifted": 3813, "k_pe": 382, "n_bits": 3431, "qber": 0.002617801047120419},
    {"index": 4, "m_sifted": 3841, "k_pe": 385, "n_bits": 3456, "qber": 0.005194805194805195},
]
for row in known_metadata:
    i = row["index"]
    for who, remote_name in [("alice", f"alice_sifted_bits_key{i}.json"), ("bob", f"bob_sifted_bits_key{i}.json")]:
        local_path = RESULTS / "data" / remote_name
        stdout, _ = bob.execute(f"cat ~/qfabric/results/{remote_name}", quiet=True)
        json.loads(stdout)  # sanity check
        local_path.write_text(stdout)
    row["alice_path"] = str(RESULTS / "data" / f"alice_sifted_bits_key{i}.json")
    row["bob_path"] = str(RESULTS / "data" / f"bob_sifted_bits_key{i}.json")
    row["alice_raw_path"] = None
    row["bob_raw_path"] = None
print("recovered keys 0-4")

In [8]:
from qne.cascade.key import key_from_sifted_json
from qne.cascade.parameter_estimation import split_pe_and_generation

recovered_5_to_9 = []
for i in range(5, 10):
    alice_raw = RESULTS / "data" / f"alice_sifted_bits_key{i}_raw.json"
    bob_raw = RESULTS / "data" / f"bob_sifted_bits_key{i}_raw.json"
    if not alice_raw.exists():
        print(f"key {i}: _raw file missing, can't recompute -- check diagnostic output")
        continue
    alice_key_full, alice_idx = key_from_sifted_json(str(alice_raw), "alice_bits")
    bob_key_full, bob_idx = key_from_sifted_json(str(bob_raw), "bob_bits")
    split = split_pe_and_generation(alice_key_full, bob_key_full, sample_fraction=0.1, seed=100000 + i)
    recovered_5_to_9.append({"index": i, "m_sifted": split["m"], "k_pe": split["k"], "n_bits": split["n"],
                              "qber": split["qber"], "alice_path": str(RESULTS / "data" / f"alice_sifted_bits_key{i}.json"),
                              "bob_path": str(RESULTS / "data" / f"bob_sifted_bits_key{i}.json")})
    print(f"key {i}: recomputed m={split['m']}, k={split['k']}, n={split['n']}, qber={split['qber']:.5f}")

key 5: recomputed m=3839, k=384, n=3455, qber=0.01042
key 6: recomputed m=3829, k=383, n=3446, qber=0.02350
key 7: recomputed m=3765, k=377, n=3388, qber=0.01061
key 8: recomputed m=3812, k=382, n=3430, qber=0.01047
key 9: recomputed m=3724, k=373, n=3351, qber=0.01340


In [9]:
import pandas as pd
combined = pd.DataFrame(known_metadata + recovered_5_to_9)
combined.to_csv(str(RESULTS / "data" / "key_pairs_metadata.csv"), index=False)
print(f"key_pairs_metadata.csv rebuilt with {len(combined)} keys")

key_pairs_metadata.csv rebuilt with 10 keys


In [1]:
# --- Step 1: diagnose. Changes nothing. ---
import sys, json
from pathlib import Path

PROJECT_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(PROJECT_DIR))
sys.path.insert(0, str(PROJECT_DIR / 'scripts'))
RESULTS = PROJECT_DIR / "results"

import deploy_fabric as deploy
fablib = deploy.get_fablib()
slice_obj = fablib.get_slice(name='qfabric-bb84-2')   # your ORIGINAL site-1 slice
bob = slice_obj.get_node("bob")

print("=== local results/ state ===")
for i in range(10):
    for who in ["alice", "bob"]:
        p = RESULTS / "data" / f"{who}_sifted_bits_key{i}.json"
        print(f"  {p.name}: {'present' if p.exists() else 'MISSING'}")
print("  key_pairs_metadata.csv:", (RESULTS / "data" / "key_pairs_metadata.csv").exists())

print("\n=== remote (bob, site-1 slice) state ===")
for i in range(10):
    stdout, _ = bob.execute(f"test -f ~/qfabric/results/alice_sifted_bits_key{i}.json && echo present || echo MISSING", quiet=True)
    print(f"  key{i} (alice file on bob node): {stdout.strip()}")

Orchestrator,orchestrator.fabric-testbed.net
Credential Manager,cm.fabric-testbed.net
Core API,uis.fabric-testbed.net
Artifact Manager,artifacts.fabric-testbed.net
CEPH Manager,https://ceph-mgr.fabric-testbed.net
Token File,/home/fabric/work/fabric_config/id_token.json
Project ID,24f4c8f3-e872-492a-9a83-b48211a91966
Bastion Host,bastion.fabric-testbed.net
Bastion Username,audreyf_0000527467
Bastion Private Key File,/home/fabric/work/fabric_config/fabric_bastion_key
Slice Public Key File,/home/fabric/work/fabric_config/slice_key.pub


User: audreyf@illinois.edu bastion key is valid!
Configuration is valid
=== local results/ state ===
  alice_sifted_bits_key0.json: MISSING
  bob_sifted_bits_key0.json: MISSING
  alice_sifted_bits_key1.json: MISSING
  bob_sifted_bits_key1.json: MISSING
  alice_sifted_bits_key2.json: MISSING
  bob_sifted_bits_key2.json: MISSING
  alice_sifted_bits_key3.json: MISSING
  bob_sifted_bits_key3.json: MISSING
  alice_sifted_bits_key4.json: MISSING
  bob_sifted_bits_key4.json: MISSING
  alice_sifted_bits_key5.json: MISSING
  bob_sifted_bits_key5.json: MISSING
  alice_sifted_bits_key6.json: MISSING
  bob_sifted_bits_key6.json: MISSING
  alice_sifted_bits_key7.json: MISSING
  bob_sifted_bits_key7.json: MISSING
  alice_sifted_bits_key8.json: MISSING
  bob_sifted_bits_key8.json: MISSING
  alice_sifted_bits_key9.json: MISSING
  bob_sifted_bits_key9.json: MISSING
  key_pairs_metadata.csv: False

=== remote (bob, site-1 slice) state ===
  key0 (alice file on bob node): present
  key1 (alice file on bo

In [3]:
import pandas as pd

# From your original snapshot -- cross-check n_bits against what Step 1 recovered above
known_metadata = [
    {"index": 0, "m_sifted": 3894, "k_pe": 390, "n_bits": 3504, "qber": 0.01282051282051282},
    {"index": 1, "m_sifted": 3788, "k_pe": 379, "n_bits": 3409, "qber": 0.0158311345646438},
    {"index": 2, "m_sifted": 3734, "k_pe": 374, "n_bits": 3360, "qber": 0.00267379679144385},
    {"index": 3, "m_sifted": 3813, "k_pe": 382, "n_bits": 3431, "qber": 0.002617801047120419},
    {"index": 4, "m_sifted": 3841, "k_pe": 385, "n_bits": 3456, "qber": 0.005194805194805195},
]
for row in known_metadata:
    i = row["index"]
    row["alice_path"] = str(RESULTS / "data" / f"alice_sifted_bits_key{i}.json")
    row["bob_path"] = str(RESULTS / "data" / f"bob_sifted_bits_key{i}.json")
    # _raw files were never uploaded remotely (only the post-PE-split generation
    # files were), so these two are genuinely unrecoverable -- left blank rather
    # than pointing at something that doesn't exist.
    row["alice_raw_path"] = None
    row["bob_raw_path"] = None

pd.DataFrame(known_metadata).to_csv(str(RESULTS / "data" / "key_pairs_metadata.csv"), index=False)
print("wrote key_pairs_metadata.csv for keys 0-4")

wrote key_pairs_metadata.csv for keys 0-4


## Section C — 5 more real keys, 5 → 10 (Feedback #9)

Extends `collect_key_pairs` with a `start_index`, so the new keys land at indices 5-9
without touching your existing `key_pairs_metadata.csv` (indices 0-4) or the `key{i}` JSON
files that Sections 4-8 of the paper already depend on. Then recomputes Table 2
(finite-key vs. asymptotic length) for all 10 keys, so `ell_finite`'s sensitivity to any
single key (key 1 at `ell=1` had to be excluded as degenerate) is easier to characterize.

In [4]:
import sys, json
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(PROJECT_DIR))
sys.path.insert(0, str(PROJECT_DIR / 'scripts'))

import deploy_fabric as deploy
from qne.cascade.key import key_from_sifted_json
from qne.cascade.parameter_estimation import split_pe_and_generation
from qne.cascade.finite_key import finite_key_output_length, asymptotic_key_length

RESULTS = PROJECT_DIR / "results"

SLICE_NAME = 'qfabric-bb84-2'  # site-1 slice, same as Section A / the original 5 keys
fablib = deploy.get_fablib()
slice_obj = fablib.get_slice(name=SLICE_NAME)
alice = slice_obj.get_node("alice")
bob = slice_obj.get_node("bob")
bob_ip = "10.10.1.2"


Orchestrator,orchestrator.fabric-testbed.net
Credential Manager,cm.fabric-testbed.net
Core API,uis.fabric-testbed.net
Artifact Manager,artifacts.fabric-testbed.net
CEPH Manager,https://ceph-mgr.fabric-testbed.net
Token File,/home/fabric/work/fabric_config/id_token.json
Project ID,24f4c8f3-e872-492a-9a83-b48211a91966
Bastion Host,bastion.fabric-testbed.net
Bastion Username,audreyf_0000527467
Bastion Private Key File,/home/fabric/work/fabric_config/fabric_bastion_key
Slice Public Key File,/home/fabric/work/fabric_config/slice_key.pub


User: audreyf@illinois.edu bastion key is valid!
Configuration is valid


In [6]:
def collect_key_pairs_from(deploy, slice_obj, alice, bob, bob_ip, project_dir,
                           start_index, n_keys, scenario_path="validation/scenarios/fabric_1km.yml",
                           pe_sample_fraction=0.1, pe_seed_base=100000):
    """Same body as qne.cascade.sweep_utils.collect_key_pairs, but starts at
    start_index and APPENDS to key_pairs_metadata.csv instead of overwriting it --
    so your existing 5 keys (indices 0-4) are preserved."""
    alice_mac = alice.get_interface(network_name="net_alice_switch").get_mac()
    bob_mac = bob.get_interface(network_name="net_switch_bob").get_mac()
    sw_alice_mac = slice_obj.get_node("switch").get_interface(network_name="net_alice_switch").get_mac()

    key_pairs = []
    for i in range(start_index, start_index + n_keys):
        print(f"\n=== Collecting key pair {i} ===")
        deploy.run_bb84(slice_obj, scenario_path, alice_mac, bob_mac,
                          sw_alice_mac=sw_alice_mac, bob_data_ip=bob_ip)

        stdout, _ = alice.execute("cat ~/qfabric/results/alice_sifted_bits.json", quiet=True)
        alice_raw_path = project_dir / "results" / "data" / f"alice_sifted_bits_key{i}_raw.json"
        alice_raw_path.write_text(stdout)

        stdout, _ = bob.execute("cat ~/qfabric/results/bob_sifted_bits.json", quiet=True)
        bob_raw_path = project_dir / "results" / "data" / f"bob_sifted_bits_key{i}_raw.json"
        bob_raw_path.write_text(stdout)

        alice_key_full, alice_idx_full = key_from_sifted_json(str(alice_raw_path), "alice_bits")
        bob_key_full, bob_idx_full = key_from_sifted_json(str(bob_raw_path), "bob_bits")

        if alice_idx_full != bob_idx_full:
            print(f"  key pair {i}: index mismatch -- skipping (raw files saved for inspection)")
            continue

        try:
            split = split_pe_and_generation(alice_key_full, bob_key_full,
                                             sample_fraction=pe_sample_fraction, seed=pe_seed_base + i)
        except ValueError as e:
            print(f"  key pair {i}: PE split failed ({e}) -- skipping")
            continue

        m, k, n = split["m"], split["k"], split["n"]
        qber_i = split["qber"]
        alice_key_i, bob_key_i = split["alice_gen"], split["bob_gen"]
        print(f"  key pair {i}: m={m} sifted, k={k} PE sample, n={n} generation bits, qber={qber_i:.4f}")

        alice_path = project_dir / "results" / "data" / f"alice_sifted_bits_key{i}.json"
        bob_path = project_dir / "results" / "data" / f"bob_sifted_bits_key{i}.json"
        gen_indices = split["gen_indices"]
        alice_matching_indices_gen = [alice_idx_full[idx] for idx in gen_indices]

        alice_path.write_text(json.dumps({"alice_bits": alice_key_i.bits.tolist(),
                                           "matching_indices": alice_matching_indices_gen}))
        bob_path.write_text(json.dumps({"bob_bits": bob_key_i.bits.tolist(),
                                         "matching_indices": alice_matching_indices_gen}))

        key_pairs.append({"index": i, "m_sifted": m, "k_pe": k, "n_bits": n, "qber": qber_i,
                           "alice_path": str(alice_path), "bob_path": str(bob_path),
                           "alice_raw_path": str(alice_raw_path), "bob_raw_path": str(bob_raw_path)})

        bob.upload_file(str(alice_path), f"qfabric/results/alice_sifted_bits_key{i}.json")
        bob.upload_file(str(bob_path), f"qfabric/results/bob_sifted_bits_key{i}.json")

    return pd.DataFrame(key_pairs)


existing_meta = pd.read_csv(str(RESULTS / "data" / "key_pairs_metadata.csv"))
new_keys_df = collect_key_pairs_from(deploy, slice_obj, alice, bob, bob_ip, PROJECT_DIR,
                                      start_index=5, n_keys=5)
combined_meta = pd.concat([existing_meta, new_keys_df], ignore_index=True)
combined_meta.to_csv(str(RESULTS / "data" / "key_pairs_metadata.csv"), index=False)
print(f"\nkey_pairs_metadata.csv now has {len(combined_meta)} keys (indices "
      f"{combined_meta['index'].min()}-{combined_meta['index'].max()})")


FileNotFoundError: [Errno 2] No such file or directory: '/home/fabric/work/qkd-dependability/results/key_pairs_metadata.csv'

In [14]:
# --- Recompute Table 2 (finite-key vs. asymptotic length) for all 10 keys ---
rows = []
for _, row in combined.iterrows():
    i, n, k, qber = int(row["index"]), int(row["n_bits"]), int(row["k_pe"]), float(row["qber"])
    ell_asymptotic = asymptotic_key_length(n, qber)
    try:
        ell_finite, r, t, nu = finite_key_output_length(n, k, qber)
    except ValueError as e:
        ell_finite = None
        print(f"key {i}: finite-key optimization failed ({e})")
    reduction = (1 - ell_finite / ell_asymptotic) * 100 if ell_finite and ell_asymptotic else None
    rows.append({"key": i, "n_bits": n, "qber": qber,
                 "ell_asymptotic": ell_asymptotic, "ell_finite": ell_finite, "reduction_pct": reduction})

table2_extended = pd.DataFrame(rows)
table2_extended.to_csv(str(RESULTS / "data" / "finitekey_vs_asymptotic_table2_10keys.csv"), index=False)
print(table2_extended.to_string(index=False))
print(f"\nOriginal 5-key mean reduction: {table2_extended.iloc[:5]['reduction_pct'].mean():.2f}%")
print(f"All-10-key mean reduction:     {table2_extended['reduction_pct'].mean():.2f}%")
print("(Exclude any key with an extreme ell_finite, e.g. <=1 bit, from the mean the same "
      "way key 1 was excluded in the original 5-key table.)")


NameError: name 'asymptotic_key_length' is not defined

In [13]:
import pandas as pd
combined = pd.DataFrame(known_metadata + recovered_5_to_9)
combined.to_csv(str(RESULTS / "data" / "key_pairs_metadata.csv"), index=False)
print(f"key_pairs_metadata.csv rebuilt with {len(combined)} keys")

key_pairs_metadata.csv rebuilt with 10 keys
